In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy
from sklearn.datasets import make_circles
import pickle
import os

# Конфигурация среды
CONFIG = {
    'homology_dims': (0, 1, 2, 3),
    'data_shape': (300, 2),
    'results_dir': 'giotto_results',
    'visualization': {
        'style': 'whitegrid',
        'palette': 'muted',
        'figsize': (12, 6)
    }
}

CONFIG.update({
    'data_dims': 5,  # Исходная размерность данных
    'proj_method': 'pca',  # pca, tsne, или None
    'max_viz_dims': 3  # Максимум 3 для 3D-визуализации
})

# Инициализация директорий
for dir_path in [CONFIG['results_dir'], 'visualizations']:
    os.makedirs(dir_path, exist_ok=True)

In [33]:
def generate_nd_data():
    """Генерирует данные произвольной размерности"""
    from sklearn.datasets import make_blobs
    data, _ = make_blobs(
        n_samples=400,
        n_features=CONFIG['data_dims'],
        centers=3,
        cluster_std=2.0
    )
    return data.astype(np.float32)

sample_data = generate_nd_data()

In [34]:
def visualize_high_d(data, title_suffix):
    """Универсальная визуализация для любых размерностей"""
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    # Проекция данных
    if data.shape[1] > CONFIG['max_viz_dims']:
        if CONFIG['proj_method'] == 'pca':
            proj = PCA(n_components=2).fit_transform(data)
            label = 'PCA проекция'
        elif CONFIG['proj_method'] == 'tsne':
            proj = TSNE(n_components=2).fit_transform(data)
            label = 't-SNE проекция'
        else:
            raise ValueError("Неверный метод проекции")
    else:
        proj = data
        label = 'Исходные данные'
    
    # Визуализация
    plt.figure(figsize=(10, 6))
    
    if proj.shape[1] == 3:
        from mpl_toolkits.mplot3d import Axes3D
        ax = plt.subplot(111, projection='3d')
        ax.scatter(
            proj[:, 0], 
            proj[:, 1], 
            proj[:, 2],
            c=sns.color_palette("hsv", proj.shape[0]),
            s=40
        )
    else:
        sns.scatterplot(
            x=proj[:, 0],
            y=proj[:, 1],
            hue=np.linalg.norm(data, axis=1),
            palette='viridis',
            s=50
        )
    
    plt.title(f'{label} | {title_suffix}')
    plt.savefig(f"visualizations/{title_suffix}.png", dpi=150)
    plt.close()


visualize_high_d(sample_data, 'Исходные данные')

In [35]:
def compute_persistence(data):
    """Вычисляет персистентные гомологии с очисткой памяти"""
    homology = VietorisRipsPersistence(
        homology_dimensions=CONFIG['homology_dims'],
        collapse_edges=True,
        n_jobs=-1
    )
    
    # Преобразование в формат [n_samples, n_points, n_features]
    reshaped_data = data.reshape(1, *data.shape)
    persistence = homology.fit_transform(reshaped_data)
    return persistence[0]

persistence_diagrams = compute_persistence(sample_data)

In [36]:
def plot_dimension_persistence(persistence_data):
    """Строит и сохраняет отдельные диаграммы для каждой размерности"""
    df = pd.DataFrame(persistence_data, columns=['birth', 'death', 'dim'])
    df['dim'] = df['dim'].astype(int)
    
    # Параметры стиля
    plot_style = {
        'size': 40,
        'edgecolor': 'black',
        'alpha': 0.7
    }
    
    # Создаем отдельный график для каждой размерности
    for dim in CONFIG['homology_dims']:
        dim_data = df[df['dim'] == dim]
        if dim_data.empty:
            continue
            
        plt.figure(figsize=(10, 6))
        sns.scatterplot(
            data=dim_data,
            x='birth',
            y='death',
            hue='dim',
            s=plot_style['size'],
            edgecolor=plot_style['edgecolor'],
            alpha=plot_style['alpha'],
            legend=False
        )
        
        # Линия диагонали и аннотации
        max_val = max(dim_data[['birth', 'death']].max().max(), 1.0)
        plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
        plt.title(f'Персистентная диаграмма (Размерность {dim})')
        plt.xlabel('Время рождения')
        plt.ylabel('Время смерти')
        
        # Сохранение в отдельный файл
        plt.savefig(f"visualizations/persistence_dim_{dim}.png", dpi=150)
        plt.close()

In [37]:
visualize_high_d(persistence_diagrams[:, :2], 'Персистентные признаки')
plot_dimension_persistence(persistence_diagrams)

In [38]:
def plot_tda_results(data, suffix):
    """Универсальная функция визуализации"""
    sns.set_style(CONFIG['visualization']['style'])
    plt.figure(figsize=CONFIG['visualization']['figsize'])
    
    if suffix == 'barcode':
        df = pd.DataFrame(persistence_diagrams, columns=['birth', 'death', 'dim'])
        df = df[df['dim'] <= max(CONFIG['homology_dims'])]
        
        for dim in CONFIG['homology_dims']:
            dim_df = df[df['dim'] == dim]
            plt.plot(
                [dim_df['birth'], dim_df['death']],
                [np.arange(len(dim_df))]*2,
                color=sns.color_palette()[dim]
            )
            
        plt.title('Персистентные баркоды')
        plt.xlabel('Значение фильтрации')
        
    elif suffix == 'persistence':
        sns.scatterplot(
            x=persistence_diagrams[:, 0],
            y=persistence_diagrams[:, 1],
            hue=persistence_diagrams[:, 2],
            palette=CONFIG['visualization']['palette']
        )
        plt.plot([0, 1], [0, 1], 'k--')
        plt.title('Персистентная диаграмма')

    plt.savefig(f"visualizations/{suffix}.png", bbox_inches='tight')
    plt.close()

# Генерация графиков
plot_tda_results(persistence_diagrams, 'barcode')
plot_tda_results(persistence_diagrams, 'persistence')

In [39]:
def save_data(obj, name):
    """Универсальная функция сохранения"""
    # Сохранение в pickle
    with open(f"{CONFIG['results_dir']}/{name}.pkl", 'wb') as f:
        pickle.dump(obj, f, protocol=4)
        
    # Сохранение числовых данных в txt
    if isinstance(obj, np.ndarray):
        np.savetxt(f"{CONFIG['results_dir']}/{name}.txt", obj)


def save_visualization_metadata(data, name):
    """Сохраняет метаданные для воспроизведения визуализаций"""
    metadata = {
        'shape': data.shape,
        'min_vals': data.min(axis=0),
        'max_vals': data.max(axis=0),
        'proj_method': CONFIG['proj_method']
    }
    with open(f"{CONFIG['results_dir']}/{name}_meta.pkl", 'wb') as f:
        pickle.dump(metadata, f)

def save_dimension_data(persistence_data):
    """Сохраняет данные и метаданные для каждой размерности"""
    df = pd.DataFrame(persistence_data, columns=['birth', 'death', 'dim'])
    
    for dim in CONFIG['homology_dims']:
        dim_df = df[df['dim'] == dim]
        if not dim_df.empty:
            # Сохранение raw data
            dim_df[['birth', 'death']].to_csv(
                f"{CONFIG['results_dir']}/persistence_dim_{dim}.csv",
                index=False
            )
            
            # Метаданные
            meta = {
                'count': len(dim_df),
                'mean_lifetime': (dim_df['death'] - dim_df['birth']).mean(),
                'max_birth': dim_df['birth'].max()
            }
            with open(f"{CONFIG['results_dir']}/meta_dim_{dim}.pkl", 'wb') as f:
                pickle.dump(meta, f)

# Вызов функции сохранения
save_dimension_data(persistence_diagrams)
# Сохраняем все артефакты
save_data(sample_data, 'source_data')
save_data(persistence_diagrams, 'persistence_diagrams')

save_visualization_metadata(sample_data, 'source_data')